# Summary Statistics VAE — Research Notebook

This notebook implements and experiments with a **Summary Statistics VAE** for exercise form analysis.

**How it works:**
1. Each rep (variable length) is compressed into a fixed 97-dim statistics vector
2. A VAE learns to reconstruct "good form" statistics
3. High reconstruction error = bad form = low similarity score

**Camera-angle invariance** comes from training on the same exercises filmed from multiple camera angles.

## Architecture
```
Input: 97 rep stats + 5 exercise one-hot = 102
Encoder: 102 → 256 → 128 → latent (mu: 32, log_var: 32)
Decoder: 37 (32 + 5) → 128 → 256 → 97
Loss: MSE + 0.5 * KL divergence
```

## 1. Setup & Imports

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Add the AI directory to path so we can import existing utilities
AI_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if AI_DIR not in sys.path:
    sys.path.insert(0, AI_DIR)

from Utils.utils.utils import (
    ANGLE_NAMES,
    compute_angle_features_2d,
    smooth_angles,
    find_rep_boundaries,
    extract_rep_angles,
)
from process_landmarks.exercise_config import EXERCISE_CONFIGS, EXERCISE_INDEX, NUM_EXERCISES
from mlp.feature_stats import compute_rep_stats, STATS_DIM

print(f"Stats dimension: {STATS_DIM}")
print(f"Angle names: {ANGLE_NAMES}")
print(f"Exercises: {list(EXERCISE_INDEX.keys())}")
print(f"PyTorch version: {torch.__version__}")

## 2. Load Training Data

Place `.npy` landmark files in `AI/training_data/{exercise}/`.

Each file should be `(T, 33, 4)` MediaPipe landmarks from a good-form video. You can get these by:
1. Running a video through the app's `/verdict` endpoint
2. Copying the `.npy` file from `AI/MediaPipe_landmarks/` into the appropriate exercise folder

In [ ]:
TRAINING_DATA_DIR = os.path.join(AI_DIR, "training_data")

def load_all_reps(data_dir):
    """Load landmark files, extract reps, return list of dicts."""
    all_reps = []
    
    for exercise_dir in sorted(os.listdir(data_dir)):
        exercise_path = os.path.join(data_dir, exercise_dir)
        if not os.path.isdir(exercise_path):
            continue
        
        exercise_idx = EXERCISE_INDEX.get(exercise_dir)
        if exercise_idx is None:
            print(f"  Skipping unknown exercise folder: {exercise_dir}")
            continue
        
        # Pick rep detection config
        if 'squat' in exercise_dir:
            config_key = 'heavy_squat'
        else:
            config_key = 'adaptive'
        exercise_config = EXERCISE_CONFIGS.get(config_key, EXERCISE_CONFIGS['heavy_squat'])
        
        npy_files = [f for f in os.listdir(exercise_path) if f.endswith('.npy')]
        print(f"  {exercise_dir}: {len(npy_files)} landmark files")
        
        for fname in npy_files:
            fpath = os.path.join(exercise_path, fname)
            landmarks = np.load(fpath)
            lm = landmarks[:, :, :3] if landmarks.shape[2] > 3 else landmarks
            
            angles = compute_angle_features_2d(lm)
            smooth = smooth_angles(angles)
            
            reps, _ = find_rep_boundaries(smooth, exercise_config)
            reps_data = extract_rep_angles(smooth, reps)
            
            for rep_angles in reps_data:
                all_reps.append({
                    'angles': rep_angles,
                    'exercise': exercise_dir,
                    'exercise_idx': exercise_idx,
                    'source_file': fname,
                    'n_frames': rep_angles.shape[0],
                })
        
        count = sum(1 for r in all_reps if r['exercise'] == exercise_dir)
        print(f"    → {count} reps extracted")
    
    return all_reps

reps = load_all_reps(TRAINING_DATA_DIR)
print(f"\nTotal reps: {len(reps)}")

if not reps:
    print("\n⚠ No training data found!")
    print(f"Place .npy files in: {TRAINING_DATA_DIR}/{{exercise}}/")
    print(f"Valid exercise folders: {list(EXERCISE_INDEX.keys())}")

## 3. Visualize Training Data

Inspect the angle signals and rep statistics to understand the data before training.

In [ ]:
# Plot angle signals for a few sample reps
if reps:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    sample_reps = reps[:min(4, len(reps))]
    
    for ax, rep in zip(axes.flat, sample_reps):
        angles = rep['angles']
        for j, name in enumerate(ANGLE_NAMES):
            if name in ['left_knee', 'right_knee', 'left_hip', 'right_hip', 'trunk_lean']:
                ax.plot(angles[:, j], label=name)
        ax.set_title(f"{rep['exercise']} — {rep['n_frames']} frames\n({rep['source_file']})")
        ax.set_xlabel("Frame")
        ax.set_ylabel("Angle (degrees)")
        ax.legend(fontsize=7)
    
    plt.suptitle("Core Angle Features per Rep", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Rep length distribution
    lengths = [r['n_frames'] for r in reps]
    plt.figure(figsize=(8, 4))
    plt.hist(lengths, bins=20, edgecolor='black', alpha=0.7)
    plt.xlabel("Frames per rep")
    plt.ylabel("Count")
    plt.title(f"Rep Length Distribution (n={len(reps)}, mean={np.mean(lengths):.0f}, std={np.std(lengths):.0f})")
    plt.show()
else:
    print("No reps to visualize — add training data first.")

## 4. Compute Summary Statistics & Prepare Training Data

Each rep `(T, 13)` is compressed to a **97-dim** vector:
- 13 angle features x 7 stats (mean, std, min, max, range, skewness, p25) = 91
- 4 symmetry stats (knee + hip mean/std)
- 2 depth stats (min left/right knee)

In [ ]:
if reps:
    # Compute stats for every rep
    stats_list = []
    exercise_indices = []
    
    for rep in reps:
        stats = compute_rep_stats(rep['angles'])
        stats_list.append(stats)
        exercise_indices.append(rep['exercise_idx'])
    
    stats_array = np.array(stats_list, dtype=np.float32)
    exercise_array = np.array(exercise_indices, dtype=np.int64)
    
    print(f"Stats matrix shape: {stats_array.shape}")  # (n_reps, 97)
    print(f"Exercise distribution: {np.bincount(exercise_array)}")
    
    # Normalize
    train_mean = stats_array.mean(axis=0)
    train_std = stats_array.std(axis=0) + 1e-8
    normalized = (stats_array - train_mean) / train_std
    
    # Exercise one-hot
    exercise_oh = np.zeros((len(exercise_array), NUM_EXERCISES), dtype=np.float32)
    for i, idx in enumerate(exercise_array):
        exercise_oh[i, idx] = 1.0
    
    print(f"Normalized stats range: [{normalized.min():.2f}, {normalized.max():.2f}]")
    print(f"One-hot shape: {exercise_oh.shape}")
else:
    print("No reps — add training data first.")

## 5. Define & Train the VAE

In [ ]:
from mlp.model import StatsVAE, vae_loss

# Hyperparameters — tweak these to experiment
EPOCHS = 200
LR = 1e-3
BATCH_SIZE = 32
BETA = 0.5        # KL divergence weight (lower = more reconstruction focus)
LATENT_DIM = 32

if reps:
    # Create model
    model = StatsVAE(stats_dim=STATS_DIM, n_exercises=NUM_EXERCISES, latent_dim=LATENT_DIM)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    
    # DataLoader
    dataset = TensorDataset(torch.tensor(normalized), torch.tensor(exercise_oh))
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    # Training loop with loss tracking
    loss_history = {'total': [], 'recon': [], 'kl': []}
    
    model.train()
    for epoch in range(EPOCHS):
        epoch_loss = 0
        epoch_recon = 0
        epoch_kl = 0
        
        for batch_stats, batch_exercise in loader:
            reconstruction, mu, log_var = model(batch_stats, batch_exercise)
            loss, recon_loss, kl_loss = vae_loss(reconstruction, batch_stats, mu, log_var, BETA)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            epoch_recon += recon_loss.item()
            epoch_kl += kl_loss.item()
        
        n_batches = len(loader)
        loss_history['total'].append(epoch_loss / n_batches)
        loss_history['recon'].append(epoch_recon / n_batches)
        loss_history['kl'].append(epoch_kl / n_batches)
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} — loss: {epoch_loss/n_batches:.6f} "
                  f"(recon: {epoch_recon/n_batches:.6f}, kl: {epoch_kl/n_batches:.6f})")
    
    print("\nTraining complete!")
else:
    print("No training data available.")

## 6. Training Loss Curves

In [ ]:
if reps:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(loss_history['total'], label='Total Loss')
    ax1.plot(loss_history['recon'], label='Reconstruction Loss')
    ax1.plot(loss_history['kl'], label=f'KL Loss (x{BETA})')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss Curves')
    ax1.legend()
    ax1.set_yscale('log')
    ax1.grid(True, alpha=0.3)
    
    # Zoom into last 50% of training
    half = len(loss_history['total']) // 2
    ax2.plot(range(half, EPOCHS), loss_history['total'][half:], label='Total')
    ax2.plot(range(half, EPOCHS), loss_history['recon'][half:], label='Recon')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.set_title('Loss Curves (Last 50%)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 7. Evaluate — Reconstruction Errors & Score Calibration

Run all training reps through the model to establish the error distribution.
This distribution is used to calibrate the sigmoid mapping from error → 0-1 similarity score.

In [ ]:
def sigmoid_calibrate(error, median, iqr):
    """Map reconstruction error to 0-1 similarity score."""
    scale = max(iqr * 2, 1e-6)
    return 1.0 / (1.0 + np.exp((error - median * 1.5) / scale))

if reps:
    model.eval()
    errors = []
    per_feature_errors = []
    
    with torch.no_grad():
        for i in range(len(normalized)):
            s = torch.tensor(normalized[i]).unsqueeze(0)
            e = torch.tensor(exercise_oh[i]).unsqueeze(0)
            recon, _, _ = model(s, e)
            
            # Overall error
            error = float(((s - recon) ** 2).mean())
            errors.append(error)
            
            # Per-feature errors (13 features, 7 stats each)
            feat_errors = []
            recon_np = recon.squeeze(0).numpy()
            for j in range(13):
                sl = slice(j * 7, (j + 1) * 7)
                feat_err = float(np.mean((normalized[i, sl] - recon_np[sl]) ** 2))
                feat_errors.append(feat_err)
            per_feature_errors.append(feat_errors)
    
    errors = np.array(errors)
    per_feature_errors = np.array(per_feature_errors)
    
    median_error = float(np.median(errors))
    iqr = float(np.percentile(errors, 75) - np.percentile(errors, 25))
    
    # Convert to similarity scores
    scores = np.array([sigmoid_calibrate(e, median_error, iqr) for e in errors])
    
    print(f"Reconstruction error distribution:")
    print(f"  Median: {median_error:.6f}")
    print(f"  IQR: {iqr:.6f}")
    print(f"  Range: [{errors.min():.6f}, {errors.max():.6f}]")
    print(f"\nCalibrated similarity scores:")
    print(f"  Mean: {scores.mean():.3f}")
    print(f"  Range: [{scores.min():.3f}, {scores.max():.3f}]")
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.hist(errors, bins=20, edgecolor='black', alpha=0.7)
    ax1.axvline(median_error, color='red', linestyle='--', label=f'Median: {median_error:.4f}')
    ax1.set_xlabel('Reconstruction Error (MSE)')
    ax1.set_ylabel('Count')
    ax1.set_title('Training Set Reconstruction Errors')
    ax1.legend()
    
    ax2.hist(scores, bins=20, edgecolor='black', alpha=0.7, color='green')
    ax2.set_xlabel('Similarity Score')
    ax2.set_ylabel('Count')
    ax2.set_title('Calibrated Similarity Scores (Training Set)')
    
    plt.tight_layout()
    plt.show()

## 8. Per-Feature Error Analysis

See which joints the VAE reconstructs well vs. poorly. This helps identify which features drive the score.

In [ ]:
if reps:
    mean_feat_errors = per_feature_errors.mean(axis=0)
    
    plt.figure(figsize=(12, 5))
    bars = plt.bar(range(13), mean_feat_errors, tick_label=ANGLE_NAMES, alpha=0.8, edgecolor='black')
    
    # Color core features differently
    core = ['left_knee', 'right_knee', 'left_hip', 'right_hip', 'trunk_lean']
    for i, name in enumerate(ANGLE_NAMES):
        if name in core:
            bars[i].set_color('coral')
        else:
            bars[i].set_color('steelblue')
    
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Mean Reconstruction Error')
    plt.title('Per-Feature Reconstruction Error (coral = core features)')
    plt.tight_layout()
    plt.show()

## 9. Camera Angle Consistency Test

If you have training data from multiple camera angles, check whether scores are consistent across angles.
Group reps by source file (each file = different camera angle) and compare score distributions.

In [ ]:
if reps:
    # Group scores by source file
    from collections import defaultdict
    file_scores = defaultdict(list)
    for i, rep in enumerate(reps):
        file_scores[rep['source_file']].append(scores[i])
    
    if len(file_scores) > 1:
        print("Scores grouped by source file (proxy for camera angle):\n")
        for fname, fscores in sorted(file_scores.items()):
            fscores = np.array(fscores)
            print(f"  {fname}: mean={fscores.mean():.3f}, std={fscores.std():.3f}, n={len(fscores)}")
        
        # Box plot
        plt.figure(figsize=(10, 5))
        labels = list(file_scores.keys())
        data = [file_scores[k] for k in labels]
        plt.boxplot(data, labels=[l[:20] for l in labels])
        plt.xticks(rotation=45, ha='right')
        plt.ylabel('Similarity Score')
        plt.title('Score Distribution per Source File (Camera Angle Consistency)')
        plt.tight_layout()
        plt.show()
    else:
        print("Only one source file — add data from multiple camera angles to test consistency.")

## 10. Export Model for Production

Save the trained model weights and calibration data to `AI/mlp/models/` so it can be used by the production system.

To activate this model in production, change `ACTIVE_MODEL = "stats_vae"` in `AI/process_landmarks/model_config.py`.

In [ ]:
import json

if reps:
    MODELS_DIR = os.path.join(AI_DIR, "mlp", "models")
    os.makedirs(MODELS_DIR, exist_ok=True)
    
    # Save model weights
    model_path = os.path.join(MODELS_DIR, "stats_vae.pt")
    torch.save(model.state_dict(), model_path)
    print(f"Saved model weights to: {model_path}")
    
    # Save normalization stats
    stats_path = os.path.join(MODELS_DIR, "norm_stats.npz")
    np.savez(stats_path, mean=train_mean, std=train_std)
    print(f"Saved normalization stats to: {stats_path}")
    
    # Save calibration config
    all_flexions = []
    for rep in reps:
        knee_inner = (rep['angles'][:, 2].min() + rep['angles'][:, 3].min()) / 2
        all_flexions.append(180.0 - knee_inner)
    training_avg_flexion = float(np.mean(all_flexions))
    
    config = {
        "median_error": median_error,
        "iqr": iqr,
        "training_avg_flexion": training_avg_flexion,
        "n_training_reps": len(reps),
        "model_type": "stats_vae",
        "epochs": EPOCHS,
        "beta": BETA,
        "latent_dim": LATENT_DIM,
    }
    config_path = os.path.join(MODELS_DIR, "config.json")
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"Saved config to: {config_path}")
    print(f"\nCalibration: median_error={median_error:.6f}, iqr={iqr:.6f}")
    print(f"Training avg flexion: {training_avg_flexion:.1f} degrees")